# Evidencia de Aprendizaje 1: Diseño e implementación de una base de datos analítica
**Estudiante:** Alvaro Enrique Muñoz Giraldo
**Dataset:** Wanderbricks (samples.wanderbricks)
**Enlace al Video de Sustentación:** [Ver Video en Google Drive](https://drive.google.com/file/d/1iglZ4R2hzvrSEsYt0fdqE3v7lMra3fp9/view?usp=sharing)

In [0]:
# Apuntando a la ruta correcta dentro de samples
print("--- VOLUMEN DE DATOS ---")
print("Reservas:", spark.table("samples.wanderbricks.bookings").count())
print("Propiedades:", spark.table("samples.wanderbricks.properties").count())
print("Clickstream:", spark.table("samples.wanderbricks.clickstream").count())

print("\n--- ESTRUCTURA DE LOS DATOS ANIDADOS (Clickstream) ---")
spark.table("samples.wanderbricks.clickstream").printSchema()

print("\n--- MUESTRA DE DATOS ---")
display(spark.table("samples.wanderbricks.clickstream").limit(5))

--- VOLUMEN DE DATOS ---
Reservas: 72247
Propiedades: 18163
Clickstream: 100000

--- ESTRUCTURA DE LOS DATOS ANIDADOS (Clickstream) ---
root
 |-- event: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- referrer: string (nullable = true)
 |-- property_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: long (nullable = true)


--- MUESTRA DE DATOS ---


event,metadata,property_id,timestamp,user_id
view,"List(desktop, email)",5135,2025-09-04T23:26:54.935Z,16390
search,"List(desktop, direct)",7805,2025-09-04T23:26:54.935Z,3655
click,"List(mobile, direct)",17349,2025-09-04T23:26:54.935Z,100094
filter,"List(tablet, ad)",6128,2025-09-04T23:26:54.935Z,102187
search,"List(mobile, google)",3438,2025-09-04T23:26:54.935Z,40863


### 2. Justificación del Modelo de Datos
Para el caso de Wanderbricks (Marketplace de alquiler vacacional), analizamos tres paradigmas:

1. **Paradigma Relacional:** Ideal para garantizar la integridad de las transacciones financieras (pagos y reservas). Sin embargo, fue **descartado** como modelo único porque sufre de rigidez ante datos semiestructurados como el `clickstream` (actividad de navegación) y los metadatos anidados de las reseñas, requiriendo normalizaciones excesivas (múltiples JOINs que degradarían el rendimiento analítico).
2. **Paradigma NoSQL (Documental):** Excelente para absorber los eventos de navegación y reseñas mutables sin un esquema estricto. **Descartado** como única solución porque carece de garantías transaccionales ACID robustas nativas para operaciones financieras complejas cruzadas (pagos vs reservas).
3. **Lakehouse (Elegido):** Es el modelo óptimo para Wanderbricks. Nos permite almacenar los grandes volúmenes crudos del `clickstream` (variedad de datos) en bajo costo, manejar estructuras anidadas de forma nativa con Spark, y al mismo tiempo, gracias a **Delta Lake**, nos otorga propiedades ACID (atomicidad y aislamiento) para las tablas transaccionales de pagos y reservas.

In [0]:
%sql
-- Crear el catálogo y esquema
CREATE CATALOG IF NOT EXISTS ea1_catalog;
USE CATALOG ea1_catalog;

CREATE SCHEMA IF NOT EXISTS wanderbricks_lakehouse;
USE SCHEMA wanderbricks_lakehouse;

-- Crear el volumen para ingesta de archivos crudos (simulación)
CREATE VOLUME IF NOT EXISTS raw_data;

In [0]:
%sql
-- Ingestamos los datos crudos directamente desde los samples a nuestra capa bronce
CREATE OR REPLACE TABLE bronze_bookings AS SELECT * FROM samples.wanderbricks.bookings;
CREATE OR REPLACE TABLE bronze_properties AS SELECT * FROM samples.wanderbricks.properties;
CREATE OR REPLACE TABLE bronze_users AS SELECT * FROM samples.wanderbricks.users;
CREATE OR REPLACE TABLE bronze_clickstream AS SELECT * FROM samples.wanderbricks.clickstream;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.functions import col

# 1. Capa Plata de Reservas (asegurándonos de limpiar nulos)
df_bookings_silver = spark.table("bronze_bookings").dropna(subset=["booking_id"])
df_bookings_silver.write.format("delta").mode("overwrite").saveAsTable("silver_bookings")

# 2. Capa Plata de Clickstream usando los nombres de columnas reales que detectó Databricks
df_clickstream_silver = spark.table("bronze_clickstream").select(
    col("user_id"),
    col("timestamp"),
    col("event"),
    col("property_id"),
    col("metadata") # Aquí están las estructuras anidadas requeridas por la rúbrica
).dropna(subset=["user_id"])

# Guardamos la tabla plata de clickstream
df_clickstream_silver.write.format("delta").mode("overwrite").saveAsTable("silver_clickstream")

# Visualizamos el resultado
display(spark.table("silver_clickstream").limit(3))

user_id,timestamp,event,property_id,metadata
16390,2025-09-04T23:26:54.935Z,view,5135,"List(desktop, email)"
3655,2025-09-04T23:26:54.935Z,search,7805,"List(desktop, direct)"
100094,2025-09-04T23:26:54.935Z,click,17349,"List(mobile, direct)"


In [0]:
%sql
-- 1. Atomicidad: Transacción de actualización. O se aplica toda, o falla toda.
UPDATE silver_bookings SET total_amount = total_amount * 1.10 WHERE status = 'COMPLETED';

-- 2. Time Travel: Consultar el historial de la tabla (DESCRIBE HISTORY)
DESCRIBE HISTORY silver_bookings;

-- Consultamos la versión anterior (antes del aumento de precio) usando VERSION AS OF 0
SELECT * FROM silver_bookings VERSION AS OF 0 LIMIT 3;

booking_id,user_id,property_id,check_in,check_out,guests_count,total_amount,status,created_at,updated_at
18158,104276,4749,2024-12-11,2024-12-15,1,879.15,confirmed,2024-12-04T05:28:55.000Z,2024-12-06T22:54:47.000Z
18159,102433,8618,2024-10-06,2024-10-10,3,451.33,pending,2024-01-05T13:02:52.000Z,2024-02-06T00:11:03.000Z
18160,43834,11881,2024-12-14,2024-12-18,3,593.59,pending,2024-11-10T15:31:49.000Z,2024-12-12T10:19:16.000Z


In [0]:
from pyspark.sql.functions import lit

# 3. Evolución de Esquema (Método infalible)
# Tomamos 1 registro real de la tabla para heredar los tipos de datos exactos (BIGINT, etc.)
df_base = spark.table("silver_bookings").limit(1)

# Le inyectamos la nueva columna a ese registro
df_evolucionado = df_base.withColumn("discount_applied", lit(True))

# Lo insertamos forzando a Delta a fusionar el esquema
df_evolucionado.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_bookings")

# Comprobamos que la tabla completa ahora tiene la nueva columna
display(spark.sql("SELECT booking_id, total_amount, discount_applied FROM silver_bookings LIMIT 3"))

booking_id,total_amount,discount_applied
18158,879.15,null
18159,451.33,null
18160,593.59,null


In [0]:
%sql
-- Consulta 1: Top 3 usuarios con más reservas
SELECT user_id, COUNT(booking_id) as total_reservas 
FROM silver_bookings GROUP BY user_id ORDER BY total_reservas DESC LIMIT 3;

-- Consulta 2: Volumen de eventos en la app por tipo de evento
SELECT event, COUNT(*) as interacciones 
FROM silver_clickstream GROUP BY event ORDER BY interacciones DESC LIMIT 5;

event,interacciones
click,25216
filter,24961
search,24934
view,24889


In [0]:
import pyspark.sql.functions as F

# Consulta 3: Promedio de monto por estado
print("Monto promedio por estado:")
display(spark.table("silver_bookings").groupBy("status") \
        .agg(F.avg("total_amount").alias("avg_amount")))

# Consulta 4: Detección de usuarios más activos en la app
print("Top 5 Usuarios más activos:")
display(spark.table("silver_clickstream").groupBy("user_id") \
        .agg(F.count("event").alias("total_events")) \
        .orderBy(F.desc("total_events")).limit(5))

# Consulta 5: Ingresos totales confirmados
print("Ingresos totales confirmados:")
display(spark.table("silver_bookings").filter(F.col("status") == "COMPLETED") \
        .agg(F.sum("total_amount").alias("revenue")))

Monto promedio por estado:


status,avg_amount
pending,555.2092636692948
cancelled,551.6753888187277
completed,555.3099086062532
confirmed,552.3887173175891


Top 5 Usuarios más activos:


user_id,total_events
37344,8
10033,7
15889,7
8776,7
16345,6


Ingresos totales confirmados:


revenue
null


### 7. Declaración y División del trabajo
* **Uso de IA:** Se utilizaron asistentes de IA (LLMs) como apoyo para estructurar ,para aplanar las variables del clickstream y estructurar consultas. Entiendo la lógica generada y seré capaz de sustentarla en el video.
* **Roles:** Al asistente, asumí el 100% del rol de Data Engineer y Data Analyst, implementando desde la arquitectura del catálogo hasta las consultas funcionales.